<a href="https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of one content page for one day.

For development and verification, I will use the mid-panel month of March 2026.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

# Get the token from Colab Secrets
token = userdata.get("HF_TOKEN")

# Keep the token out of the notebook code
os.environ["HF_TOKEN"] = token

# Connect to DuckDB
con = duckdb.connect()

print("DuckDB connected.")

DuckDB connected.


In [3]:
# Give DuckDB access to the Hugging Face token securely
con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [4]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_page_days
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result = con.sql(query).df()
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_page_days
0,9841378,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Features

I will use the following five page-level signals as candidate features:

- `gsc_impressions` — measures search visibility and demand.
- `gsc_clicks` — measures clicks received from search.
- `gsc_avg_position` — measures the page's average search position.
- `ga4_sessions` — measures website traffic.
- `ga4_engaged_sessions` — measures engaged website sessions.

These are observable signals that can help describe a page's demand, search performance, and engagement.

### Label / proxy

The review-priority proxy will be derived from page-level performance signals in the warehouse data.

I will define this proxy later using the available warehouse data rather than relying on `trend_direction`, which is not a field in the daily warehouse table.

### Context

- `report_date`
- `month`
- `client_hash_id`
- `content_hash_id`
- `gsc_data_available`
- `ga4_data_available`

These help identify, group, or understand the data rather than directly representing the page's review priority.

### Excluded

- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`

I am excluding these from the first five features because they are additional traffic-source signals, while the selected features already cover search visibility, clicks, overall traffic, and engagement.

I can reconsider these fields later if validation shows that they add useful information.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the fields we are considering for Lane 2
df = con.sql("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()


df[selected_fields].head()
selected_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "gsc_data_available",
    "ga4_data_available"
]

df[selected_fields].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,gsc_data_available,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,True,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,True,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,True,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,True,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,True,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification results

- The grain check confirms that each row represents one unique page-day.
- The March 2026 slice contains data from March 1 through March 31.
- GSC and GA4 availability are checked separately using `IS TRUE`.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 1. Grain
grain_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id)
        AS unique_page_days
FROM read_parquet('{march_path}')
"""

grain_result = con.sql(grain_query).df()

# 2. Row count + date span
window_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{march_path}')
"""

window_result = con.sql(window_query).df()

# 3. Availability
availability_query = f"""
SELECT
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{march_path}')
"""

availability_result = con.sql(availability_query).df()

print("1. Grain")
display(grain_result)

print("2. Row count and date span")
display(window_result)

print("3. Availability")
display(availability_result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Grain


,total_rows,unique_page_days
0,9841378,9841378


2. Row count and date span


,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


3. Availability


,gsc_available_rows,ga4_available_rows
0,3611061,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset has some limitations:

- **Unbalanced history:** Not every page has the same amount of historical data, so some pages may have less information available than others.
- **GSC-only early rows:** Some rows have GSC data available while GA4 data is missing, so GA4-based features cannot be used equally for every page.
- **Window overlaps:** When creating features and labels from different time windows, the windows may overlap. I need to make sure that information from outside the decision window is not used when building features.

Because of these limitations, model results and page-priority rankings should be interpreted as decision-support rather than guaranteed outcomes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.